<a href="https://colab.research.google.com/github/minyi-k03/Large-Language-Model-LLM-/blob/Fine-Tuning/Llama3_1_%EB%8B%A4%EA%B5%AD%EC%96%B4_%ED%85%8C%EC%8A%A4%ED%8A%B8_MGSM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Llama 3.1 다국어 성능 테스트하기 (Multilingual Grade School Math Benchmark (MGSM) 데이터셋)
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## Reference : https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
## Multilingual MGSM 데이터 : https://huggingface.co/datasets/juletxara/mgsm

In [ ]:
!nvidia-smi

# 라이브러리 설치

In [ ]:
# [Cell 1] Environment Setup for Llama-3.1
import torch

# 1. GPU Check
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found. Check Runtime type.")

print("\nInstalling Libraries for Llama-3.1...")

# 2. Install Libraries
!pip install -U "transformers>=4.43.0" "accelerate" "bitsandbytes>=0.45.0" "huggingface_hub"

print("\nSetup Completed.")

# Llama 3.1 모델 불러오기

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Hugging Face Token 설정
os.environ['HF_TOKEN'] = "Input Your Token"


# 2. 모델 및 토크나이저 로드 설정
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# [중요] T4 GPU(16GB) 메모리 터짐 방지를 위한 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16  # T4는 bfloat16보다 float16이 안정적
)

print(f"Loading Model: {model_id}...")

# 토크나이저 로드
llama3_1_tokenizer = AutoTokenizer.from_pretrained(model_id)

# Llama 모델의 Padding Token 이슈 방지
if llama3_1_tokenizer.pad_token is None:
    llama3_1_tokenizer.pad_token = llama3_1_tokenizer.eos_token

# 모델 로드 (양자화 적용)
llama3_1_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config, # 설정한 4bit config 적용
    device_map="auto",
)

print(f"Model Loaded Successfully on {llama3_1_model.device}")

In [ ]:
def generate_response(system_message, user_message, tokenizer, model):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.6,
        top_p=0.9
    )
    response = outputs[0][input_ids.shape[-1]:]

    return tokenizer.decode(response, skip_special_tokens=True)

In [ ]:
print(llama3_1_model.device)

# MGSM 데이터셋을 이용한 LLM들의 다국어 추론 성능 비교

In [ ]:
# 한국어
llama3_1_inference_result = generate_response(system_message="",
                             user_message="메리는 열렬한 정원사입니다. 어제 그녀는 좋아하는 식물원에서 18개의 새로운 화분을 받았습니다. 그녀는 이미 큰 시골집의 40개의 창문 선반마다 2개의 화분을 가지고 있습니다. 그녀는 후하게도 내일 각 선반에서 하나씩의 화분을 친구들과 가족에게 주기로 결정했습니다. 메리에게 남아 있는 화분의 개수는 몇 개일까요?",
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama3_1_inference_result)
# 정답 : 58

In [ ]:
# Llama 3.0

# Mary already has 40 x 2 = 80 flower pots on the shelves of her large countryside house.

# She received 18 new flower pots yesterday.

# So, in total, she has 80 + 18 = 98 flower pots.

# She decided to give one pot from each shelf to her friends and family, which means she will give away 40 pots (since there are 40 shelves).

# So, she will have 98 - 40 = 58 flower pots left.

In [ ]:
# GPT-4o

# 메리는 40개의 창문 선반마다 2개의 화분을 가지고 있으므로, 전체 창문 선반의 화분 개수는 다음과 같습니다:

# 40
# ×
# 2
# =
# 80
# 개
# 40×2=80개
# 메리는 어제 식물원에서 18개의 새로운 화분을 받았으므로, 그녀가 가진 총 화분의 개수는 다음과 같습니다:

# 80
# +
# 18
# =
# 98
# 개
# 80+18=98개
# 그런데 메리는 내일 각 창문 선반에서 하나씩의 화분을 친구들과 가족에게 주기로 했습니다. 창문 선반은 40개 있으므로, 그녀가 주는 화분의 개수는 다음과 같습니다:

# 40
# ×
# 1
# =
# 40
# 개
# 40×1=40개
# 따라서 메리에게 남아 있는 화분의 개수는 다음과 같습니다:

# 98
# −
# 40
# =
# 58
# 개
# 98−40=58개
# 결론적으로, 메리에게 남아 있는 화분의 개수는 58개입니다.

In [ ]:
# Clova X

# 메리는 40개의 창문 선반마다 2개의 화분을 가지고 있으므로 총 2*40 = 80개의 화분을 가지고 있습니다.
# 그녀는 어제 18개의 새로운 화분을 받았으므로 총 80+18 = 98개의 화분을 가지고 있습니다.
# 그녀는 내일 각 선반에서 하나씩의 화분을 주기로 결정했으므로 총 40개의 화분을 줄 것입니다.
# 따라서 그녀에게 남아 있는 화분의 개수는 98-40 = 58개입니다.

In [ ]:
# Gemini

# 문제를 풀어봅시다.
# 1단계: 메리가 처음 가지고 있던 화분 수 계산

# 큰 시골집에 40개의 창문 선반이 있습니다.
# 각 선반에 2개의 화분이 있었으므로, 총 40 x 2 = 80개의 화분을 가지고 있었습니다.
# 2단계: 새로 받은 화분 더하기

# 어제 식물원에서 18개의 새로운 화분을 받았으므로, 총 80 + 18 = 98개의 화분이 되었습니다.
# 3단계: 나눠준 화분 빼기

# 내일 각 선반에서 하나씩의 화분을 주므로, 총 40개의 화분을 나눠줍니다.
# 따라서, 98 - 40 = 58개의 화분이 남게 됩니다.
# 답: 메리에게 남아 있는 화분의 개수는 58개입니다.

# 정리
# 메리는 후한 마음으로 친구들과 가족에게 화분을 나눠주고도 여전히 58개의 화분을 가지고 있게 됩니다. 그녀의 정원은 더욱 풍성해졌겠네요!

In [ ]:
# GPT-4o mini

# 메리가 남아 있는 화분의 개수를 계산하기 위해서는 다음과 같은 단계로 문제를 해결할 수 있습니다.

# 현재 화분의 총 개수 계산:

# 메리는 시골집의 40개 창문 선반 각각에 2개의 화분을 가지고 있습니다.
# 따라서, 현재의 화분 개수는
# 40
#  선반
# ×
# 2
#  화분/선반
# =
# 80
#  화분
# 40 선반×2 화분/선반=80 화분입니다.
# 새로 받은 화분의 개수 추가:

# 메리는 추가로 18개의 화분을 받았습니다.
# 따라서, 전체 화분의 개수는
# 80
#  화분
# +
# 18
#  화분
# =
# 98
#  화분
# 80 화분+18 화분=98 화분입니다.
# 친구들과 가족에게 줄 화분의 개수 계산:

# 메리는 각 선반에서 1개의 화분씩 친구들과 가족에게 주기로 결정했습니다.
# 전체 선반의 수는 40개이므로, 주는 화분의 개수는 40개입니다.
# 남아 있는 화분의 개수 계산:

# 전체 화분의 개수에서 친구들과 가족에게 줄 화분의 개수를 빼면 됩니다.
# 남아 있는 화분의 개수는
# 98
#  화분
# −
# 40
#  화분
# =
# 58
#  화분
# 98 화분−40 화분=58 화분입니다.
# 따라서, 메리에게 남아 있는 화분의 개수는 58개입니다.

In [ ]:
# 영어
llama3_1_inference_result = generate_response(system_message="",
                             user_message="Mary is an avid gardener. Yesterday, she received 18 new potted plants from her favorite plant nursery. She already has 2 potted plants on each of the 40 window ledges of her large country home. Feeling generous, she has decided that she will give 1 potted plant from each ledge to friends and family tomorrow. How many potted plants will Mary remain with?",
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama3_1_inference_result)
# 정답 : 58

In [ ]:
# Llama 3.0

# Mary has 2 potted plants on each of the 40 window ledges. So she has 2 x 40 = 80 potted plants. She will give 1 potted plant from each of the 40 window ledges. So she will give away 40 potted plants. She will have 80 - 40 = 40 potted plants left. She also received 18 new potted plants. So she will have 40 + 18 = 58 potted plants left.

In [ ]:
# GPT-4o

# To solve this problem, let's break it down into steps:

# Calculate the total number of potted plants Mary initially has:

# She has 2 potted plants on each of the 40 window ledges.
# The total number of potted plants on the ledges is:
# 2
# ×
# 40
# =
# 80
#  potted plants
# 2×40=80 potted plants
# Additionally, she received 18 new potted plants from the nursery.
# The total number of potted plants she has initially is:
# 80
# +
# 18
# =
# 98
#  potted plants
# 80+18=98 potted plants
# Calculate the number of potted plants she gives away:

# She gives 1 potted plant from each of the 40 window ledges.
# The total number of potted plants given away is:
# 1
# ×
# 40
# =
# 40
#  potted plants
# 1×40=40 potted plants
# Calculate the number of potted plants Mary will remain with:

# Subtract the number of potted plants given away from the total number of potted plants:
# 98
# −
# 40
# =
# 58
#  potted plants
# 98−40=58 potted plants
# Thus, Mary will remain with 58 potted plants.

In [ ]:
# Clova X

# Mary currently has 2 potted plants on each of the 40 window ledges, so she has a total of 2 * 40 = 80 potted plants.

# She plans to give away 1 potted plant from each ledge, so she will give away 40 potted plants.

# Therefore, Mary will remain with 80 - 40 = 40 potted plants.

In [ ]:
# Gemini

# Mary initially had 40 window ledges * 2 plants/ledge = 80 potted plants.
# She received 18 new potted plants, so she had 80 + 18 = 98 potted plants.
# She will give away 40 potted plants (1 from each ledge), so she will have 98 - 40 = 58 potted plants remaining.
# Therefore, Mary will remain with 58 potted plants.

In [ ]:
# GPT-4o mini

# To determine how many potted plants Mary will have left after giving some away, we need to follow these steps:

# Calculate the total number of potted plants Mary currently has:

# Mary has 2 potted plants on each of the 40 window ledges. Therefore, the total number of potted plants is:

# 2
#  plants/ledge
# ×
# 40
#  ledges
# =
# 80
#  plants
# 2 plants/ledge×40 ledges=80 plants
# Add the 18 new potted plants Mary received:

# Adding these new plants to her existing ones:

# 80
#  plants
# +
# 18
#  new plants
# =
# 98
#  plants
# 80 plants+18 new plants=98 plants
# Determine how many potted plants Mary will give away:

# Mary will give 1 potted plant from each of the 40 ledges. Therefore, she will give away:

# 1
#  plant/ledge
# ×
# 40
#  ledges
# =
# 40
#  plants
# 1 plant/ledge×40 ledges=40 plants
# Subtract the number of potted plants given away from the total number Mary had:

# To find out how many potted plants Mary will have left, subtract the number of plants given away from the total:

# 98
#  plants
# −
# 40
#  plants
# =
# 58
#  plants
# 98 plants−40 plants=58 plants
# Thus, after giving away the plants, Mary will remain with:

# 58

In [ ]:
# 일본어
llama3_1_inference_result = generate_response(system_message="",
                             user_message="マリーは熱心な演芸家です。昨日、彼女はお気に入りの育種場から18個の新しい鉢植えを受け取りました。彼女はすでに、彼女の広い田舎の家の、40の窓枠にそれぞれ2つの鉢植えを所有している。気が大きくなった彼女は、各窓枠から、1つの鉢植えを友達や家族に明日あげることにした。マリーにはいくつの鉢植えが残る？",
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama3_1_inference_result)
# 정답 : 58

In [ ]:
# Llama 3.0

# Let's break it down step by step!

# Marie already has 40 windows with 2 pots each, so she has a total of:

# 40 windows x 2 pots/window = 80 pots

# She received 18 new pots yesterday, so she now has a total of:

# 80 pots + 18 pots = 98 pots

# She decided to give away 1 pot from each of the 40 windows, so she will give away:

# 40 windows x 1 pot/window = 40 pots

# After giving away the 40 pots, Marie will be left with:

# 98 pots - 40 pots = 58 pots

# So, Marie will have 58 pots left.

In [ ]:
# GPT-4o

# マリーが現在所有している鉢植えの数を計算しましょう。

# まず、彼女はもともと40の窓枠に2つずつ鉢植えを持っていました。これにより、最初に持っていた鉢植えの数は次のようになります。

# 40
# ×
# 2
# =
# 80
#  個
# 40×2=80 個
# 次に、彼女は18個の新しい鉢植えを購入しました。これにより、彼女が現在所有している鉢植えの総数は次のようになります。

# 80
# +
# 18
# =
# 98
#  個
# 80+18=98 個
# その後、各窓枠から1つの鉢植えを取り、友達や家族にあげることにしました。40の窓枠から1つずつ鉢植えをあげるので、あげる鉢植えの数は次のようになります。

# 40
#  個
# 40 個
# 最終的にマリーに残る鉢植えの数は次のようになります。

# 98
# −
# 40
# =
# 58
#  個
# 98−40=58 個
# したがって、マリーには58個の鉢植えが残ります。

In [ ]:
# Clova X

# マリーはすでに40の窓枠にそれぞれ2つの鉢植えを所有しているため、持っている鉢植えの数は 40 * 2 = 80個です。

# 彼女は各窓枠から1つの鉢植えを友達や家族にあげると決めたので、友達や家族にあげる鉢植えの数は 40 * 1 = 40個です。

# マリーに残る鉢植えの数は 80 - 40 = 40個です。

In [ ]:
# Gemini

# マリーは、庭いじりが大好きですね！ 昨日、新しい鉢植えを手に入れたことで、さらに植物への情熱が高まっているようです。

# マリーはもともと、自宅の40個の窓枠に2つずつ、計80個の鉢植えを飾っていました。 そして今回、育種場から18個の新しい鉢植えを手に入れたので、合計で98個の鉢植えになりました。

# マリーは友達や家族に窓枠に飾ってある鉢植えを1つずつ、計40個あげるそうなので、残りは98個 - 40個 = 58個の鉢植えになります。

# よって、マリーは58個の鉢植えを残すことになります。

In [ ]:
# GPT-4o mini

# マリーは現在、40の窓枠にそれぞれ2つの鉢植えを所有しています。つまり、合計で
# 40
# ×
# 2
# =
# 80
# 40×2=80 個の鉢植えがあります。

# 彼女は、昨日、新しく18個の鉢植えを受け取りましたので、現在の鉢植えの総数は
# 80
# +
# 18
# =
# 98
# 80+18=98
# 個です。

# 彼女は、各窓枠から1つずつ鉢植えを友達や家族にあげる予定です。窓枠の数は40なので、彼女があげる鉢植えの数は40個です。

# そのため、残る鉢植えの数は
# 98
# −
# 40
# =
# 58
# 98−40=58
# 個です。

# つまり、マリーには58個の鉢植えが残ります。

In [ ]:
# 중국어
llama3_1_inference_result = generate_response(system_message="",
                             user_message="玛丽是一位充满热情的园丁。昨天，她从自己最喜欢的苗圃收到了 18 盆盆栽植物。在她大乡村住宅的 40 个窗台中，每个窗台已经有 2 盆盆栽植物。怀着慷慨之情，她决定明天从每个窗台上拿 1 盆盆栽植物送给朋友和家人。玛丽还剩下多少盆盆栽植物？",
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama3_1_inference_result)
# 정답 : 58

In [ ]:
# Llama 3.0

# 玛丽最初有 18 盆盆栽植物。每个窗台上有 2 盆盆栽植物，共有 40 个窗台，因此玛丽共有 2 x 40 = 80 盆盆栽植物。玛丽决定从每个窗台上拿 1 盆盆栽植物，共拿走 40 盆盆栽植物。剩下的盆栽植物是 18 - 40 = -22。

# 由于玛丽不能有负的盆栽植物，所以她实际上拿走了 18 盆盆栽植物。剩下的盆栽植物是 18。

In [ ]:
# GPT-4o

# 我们来一步步解答这个问题。

# 玛丽最初的盆栽数量：

# 她收到的盆栽植物数量是 18 盆。
# 她现在窗台上的盆栽数量：

# 她有 40 个窗台，每个窗台上有 2 盆盆栽植物。
# 所以窗台上的盆栽总数是
# 40
# ×
# 2
# =
# 80
# 40×2=80 盆。
# 她计划送出的盆栽数量：

# 玛丽决定从每个窗台上拿 1 盆盆栽植物，所以她将送出的盆栽数量是 40 盆。
# 剩下的盆栽数量：

# 她将送出的盆栽数量是 40 盆，这些盆栽会从窗台上的 80 盆中拿走。
# 因此，她剩下的盆栽数量是原本的 18 盆加上窗台上剩下的 40 盆，计算如下：
# 剩下的盆栽数量
# =
# 18
# +
# (
# 80
# −
# 40
# )
# =
# 18
# +
# 40
# =
# 58
#  盆
# 剩下的盆栽数量=18+(80−40)=18+40=58 盆
# 所以，玛丽还剩下 58 盆盆栽植物。

In [ ]:
# Clova X

# 玛丽从每个窗台上拿 1 盆盆栽植物送给朋友和家人,共拿出了 40 * 1 = 40 盆盆栽植物。

# 她从自己最喜欢的苗圃收到了 18 盆盆栽植物,因此她还剩下 18 - 40 = -22 盆盆栽植物。

# 由于数量为负,这是不可能的,因此我们可以推断出,玛丽收到的盆栽数量可能有误,或者她的计算有误。

In [ ]:
# Gemini

# 好的，我们来一起算一下玛丽还剩下多少盆栽植物。

# 第一步：算出玛丽现在有多少盆栽植物

# 玛丽昨天新收到了 18 盆。
# 每个窗台已经有 2 盆，而她有 40 个窗台，所以她家原本就有 2盆/个窗台 * 40 个窗台 = 80 盆。
# 所以，现在玛丽一共有 18盆（新买的） + 80盆（原有的） = 98盆。
# 第二步：算出玛丽明天要送出去多少盆

# 玛丽要从每个窗台上拿 1 盆送人，而她有 40 个窗台，所以她要送出去 40盆。
# 第三步：算出玛丽最后还剩多少盆

# 玛丽最后还剩 98盆（一共的） - 40盆（送出去的） = 58盆。
# 所以，玛丽最后还剩 58 盆盆栽植物。

In [ ]:
# GPT-4o mini

# 玛丽最初有 18 盆盆栽植物，她在 40 个窗台上，每个窗台已经放有 2 盆植物。因此，她的窗台上总共有：

# 40
#  个窗台
# ×
# 2
#  盆植物/窗台
# =
# 80
#  盆植物
# 40 个窗台×2 盆植物/窗台=80 盆植物

# 玛丽决定从每个窗台上拿走 1 盆植物送给朋友和家人，所以她总共会从 40 个窗台上拿走：

# 40
#  个窗台
# ×
# 1
#  盆植物/窗台
# =
# 40
#  盆植物
# 40 个窗台×1 盆植物/窗台=40 盆植物

# 玛丽一开始有 18 盆新植物，她将这些植物放到窗台上后，窗台上的植物总数将是：

# 80
#  盆植物
# +
# 18
#  盆新植物
# =
# 98
#  盆植物
# 80 盆植物+18 盆新植物=98 盆植物

# 在送出 40 盆植物后，剩下的植物数是：

# 98
#  盆植物
# −
# 40
#  盆植物
# =
# 58
#  盆植物
# 98 盆植物−40 盆植物=58 盆植物

# 因此，玛丽还剩下 58 盆盆栽植物。

In [ ]:
# 태국어
llama3_1_inference_result = generate_response(system_message="",
                             user_message="แมรีเป็นผู้ที่รักการทำสวนเป็นชีวิตจิตใจ เมื่อวานนี้เธอได้รับต้นไม้ใหม่ 18 กระถางจากโรงเพาะที่เธอชื่นชอบ ตามขอบหน้าต่าง 40 บานที่บ้านในชนบทของเธอมีต้นไม้ตั้งอยู่แล้วบานละ 2 กระถาง ด้วยความรู้สึกอยากแบ่งปัน เธอได้ตัดสินใจจะมอบต้นไม้ 1 กระถางบนขอบหน้าต่างแต่ละบานให้เพื่อนๆ และครอบครัวในวันพรุ่งนี้ แมรีจะเหลือต้นไม่กี่กระถาง",
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama3_1_inference_result)
# 정답 : 58

In [ ]:
# Llama 3.0

# แมรีได้รับต้นไม้ใหม่ 18 กระถาง จากโรงเพาะที่เธอชื่นชอบ เมื่อวานนี้ ตามขอบหน้าต่าง 40 บานที่บ้านในชนบทของเธอมีต้นไม้ตั้งอยู่แล้วบานละ 2 กระถาง ดังนั้นรวมทั้งหมดจะมีต้นไม้ 40 x 2 = 80 กระถาง

# แมรีได้ตัดสินใจจะมอบต้นไม้ 1 กระถางบนขอบหน้าต่างแต่ละบานให้เพื่อนๆ และครอบครัวในวันพรุ่งนี้ ซึ่งจะใช้ไป 40 บาน (ขอบหน้าต่างทั้งหมด)

# แมรีจะเหลือต้นไม้ 18 - 40 = 18 - 40 = -22 กระถาง (ไม่เหลือต้นไม้เลย เพราะได้ใช้ไปมากกว่าต้นไม้ที่มี)

In [ ]:
# GPT-4o

# เราสามารถคำนวณจำนวนต้นไม้ที่แมรีจะเหลือได้ด้วยขั้นตอนดังนี้:

# แมรีมีต้นไม้ที่ขอบหน้าต่าง 40 บาน โดยแต่ละบานมีต้นไม้ 2 กระถาง ดังนั้นจึงมีต้นไม้รวมทั้งหมดที่ขอบหน้าต่างเท่ากับ:

# 40
# ×
# 2
# =
# 80
#  กระถาง
# 40×2=80 กระถาง
# แมรีได้รับต้นไม้ใหม่ 18 กระถางจากโรงเพาะ ดังนั้นต้นไม้รวมทั้งหมดที่แมรีมีคือ:

# 80
# +
# 18
# =
# 98
#  กระถาง
# 80+18=98 กระถาง
# แมรีวางแผนที่จะมอบต้นไม้ 1 กระถางต่อขอบหน้าต่างแต่ละบานให้เพื่อนๆ และครอบครัว ซึ่งขอบหน้าต่างมีทั้งหมด 40 บาน ดังนั้นต้นไม้ที่แมรีจะมอบให้คือ:

# 40
#  กระถาง
# 40 กระถาง
# เมื่อนำต้นไม้ที่แมรีมอบให้หักลบจากต้นไม้ทั้งหมดที่เธอมี เราจะได้จำนวนต้นไม้ที่เหลือ:

# 98
# −
# 40
# =
# 58
#  กระถาง
# 98−40=58 กระถาง
# ดังนั้น แมรีจะเหลือต้นไม้ทั้งหมด 58 กระถาง หลังจากมอบให้เพื่อนๆ และครอบครัวแล้ว

In [ ]:
# Clova X

# แมรีมีต้นไม้ใหม่ 18 กระถาง และมีต้นไม้อยู่บนขอบหน้าต่าง 40 บาน โดยบานละ 2 กระถาง ดังนั้นจะมีต้นไม้ทั้งหมด 80 กระถาง

# เธอตัดสินใจมอบต้นไม้ 1 กระถางให้เพื่อนและครอบครัวที่ขอบหน้าต่างแต่ละบาน ดังนั้นจะมอบไปทั้งหมด 40 กระถาง

# ต้นไม้ที่เหลือคือ 80 - 40 = 40 กระถาง

In [ ]:
# Gemini

# ตอนนี้ฉันยังช่วยในเรื่องนั้นไม่ได้ แม้ว่าฉันจะได้รับการฝึกฝนให้ตอบคำถามอย่างถูกต้องเท่าที่สามารถทำได้ แต่ก็อาจผิดพลาดได้ในบางครั้ง ในระหว่างที่ฉันกำลังเรียนรู้ว่าจะให้ข้อมูลเกี่ยวกับการเลือกตั้งและการเมืองอย่างถูกต้องที่สุดได้อย่างไร คุณสามารถใช้ Google Search เพื่อค้นหาข้อมูลได้

In [ ]:
# GPT-4o mini

# เมื่อต้นไม้ใหม่ที่แมรีได้รับมีทั้งหมด 18 กระถาง และเธอมีต้นไม้ตั้งอยู่แล้วบนขอบหน้าต่าง 40 บาน บานละ 2 กระถาง รวมเป็นต้นไม้ทั้งหมด 40 x 2 = 80 กระถาง

# เธอจะมอบต้นไม้ 1 กระถางจากแต่ละบานให้เพื่อนและครอบครัว ซึ่งหมายความว่าเธอจะมอบต้นไม้ทั้งหมด 40 กระถาง

# ต้นไม้ที่แมรีมอบให้ไปจะทำให้ต้นไม้ที่เหลืออยู่ของเธอมีจำนวน:

# 80 (ต้นไม้ที่มีเดิม) + 18 (ต้นไม้ใหม่) - 40 (ที่มอบให้) = 58 กระถาง

# ดังนั้น แมรีจะเหลือต้นไม้ทั้งหมด 58 กระถางหลังจากมอบให้เพื่อนๆ และครอบครัว.

In [ ]:
# 벵골어
llama3_1_inference_result = generate_response(system_message="",
                             user_message="মেরি একজন উৎসুক বাগানী। গতকাল, তিনি তাঁর প্রিয় চারাগাছের নার্সারি থেকে 18টি নতুন চারাগাছের টব পেয়েছেন। ইতিমধ্যেই তাঁর সুবিশাল বাড়ির 40টি জানালার ধারের প্রতিটিতে 2টি করে চারাগাছের টব রয়েছে। উদারতা অনুভব করে, তিনি ঠিক করেছেন প্রতিটি জানালার ধার থেকে 1টি করে চারাগাছের টব আগামীকাল বন্ধুদের ও পরিবারের সদস্যদের দেবেন। মেরির কাছে কতগুলি চারাগাছের টব অবশিষ্ট থাকবে?",
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama3_1_inference_result)
# 정답 : 58

In [ ]:
# Llama 3.0

# মেরি প্রথমে 18টি নতুন চারাগাছের টব পেয়েছেন। তাঁর সুবিশাল বাড়ির 40টি জানালার ধারের প্রতিটিতে 2টি করে চারাগাছের টব রয়েছে। অর্থাৎ, মেরির কাছে মোট 40 x 2 = 80টি চারাগাছের টব রয়েছে।

# তিনি প্রতিটি জানালার ধার থেকে 1টি করে চারাগাছের টব দেবে

In [ ]:
# GPT-4o

# মেরির কাছে কতগুলি চারাগাছের টব অবশিষ্ট থাকবে তা নির্ণয় করতে নিম্নলিখিত ধাপগুলি অনুসরণ করা যেতে পারে:

# মেরির বর্তমান চারাগাছের টবের সংখ্যা নির্ণয়:

# তার 40টি জানালার ধারের প্রতিটিতে 2টি করে চারাগাছের টব রয়েছে। সুতরাং, তার বর্তমান টবের সংখ্যা হবে:
# 40
# ×
# 2
# =
# 80
# টি টব
# 40×2=80টি টব
# মেরি নতুন 18টি টব পেয়েছেন:

# সুতরাং, মেরির বর্তমান টবের সংখ্যা হবে:
# 80
# +
# 18
# =
# 98
# টি টব
# 80+18=98টি টব
# মেরি প্রতিটি জানালার ধার থেকে 1টি করে চারাগাছের টব দেবেন:

# মোট জানালা 40টি, সুতরাং মেরি মোট টব দেবেন:
# 40
# ×
# 1
# =
# 40
# টি টব
# 40×1=40টি টব
# মেরির কাছে অবশিষ্ট টবের সংখ্যা:

# সুতরাং মেরির কাছে অবশিষ্ট থাকবে:
# 98
# −
# 40
# =
# 58
# টি টব
# 98−40=58টি টব
# মেরির কাছে অবশিষ্ট থাকবে 58টি চারাগাছের টব।

In [ ]:
# Clova X

# মেরি গতকাল নার্সারি থেকে 18টি নতুন চারাগাছের টব পেয়েছেন। তাঁর বাড়ির 40টি জানালার ধারের প্রতিটিতে 2টি টব রয়েছে। তাই মেরির কাছে মোট ৮০টি টব থাকে।

# আগামীকাল প্রতিটি জানালার ধার থেকে 1টি টব দেবেন। তাহলে তিনি ৪০টি টব দেবেন।

# ৮০ - ৪০ = ৪০

# মেরির কাছে ৪০টি চারাগাছের টব অবশিষ্ট থাকবে।

In [ ]:
# Gemini

# আসুন মেরির কাছে কতগুলি চারাগাছের টব অবশিষ্ট থাকবে তা একসাথে গণনা করি।

# মেরি প্রথমে পেয়েছেন: 18টি নতুন চারাগাছের টব
# মেরির বাড়িতে ইতিমধ্যে ছিল: 40টি জানালার ধারে, প্রতিটিতে 2টি করে টব, অর্থাৎ মোট 40 * 2 = 80টি টব
# মেরির মোট টব হয়েছে: 18 + 80 = 98টি টব
# মেরি দিচ্ছেন: প্রতিটি জানালা থেকে 1টি করে, অর্থাৎ মোট 40টি টব
# অবশিষ্ট থাকবে: 98 - 40 = 58টি টব
# তাহলে, মেরির কাছে 58টি চারাগাছের টব অবশিষ্ট থাকবে।

# এই সমস্যাটি সমাধান করতে আমরা প্রথমে মেরির কাছে মোট কতগুলি টব আছে তা বের করেছি। তারপর তিনি কতগুলি টব দিচ্ছেন তা বিয়োগ করে অবশিষ্ট টবের সংখ্যা বের করেছি।

In [ ]:
# GPT-4o mini

# মেরির কাছে মোট 18টি নতুন চারাগাছের টব রয়েছে।

# তিনি ইতিমধ্যেই তাঁর বাড়ির 40টি জানালার ধারে প্রতিটি জানালায় 2টি করে চারাগাছের টব রেখেছেন। তাই মোট টবের সংখ্যা হবে:

# 40
# ×
# 2
# =
# 80
# টি টব
# 40×2=80টি টব

# এখন, তিনি প্রতিটি জানালার ধার থেকে 1টি করে টব আগামীকাল বন্ধুদের ও পরিবারের সদস্যদের দেবেন। 40টি জানালার ধার থেকে 1টি করে 40টি টব দেওয়া হবে।

# তাহলে, 40টি টব দেওয়া হলে:

# 80
# −
# 40
# =
# 40
# টি টব অবশিষ্ট থাকবে
# 80−40=40টি টব অবশিষ্ট থাকবে

# মেরির কাছে 18টি নতুন চারাগাছের টব ছিল। 40টি টব মোটে 18টি নতুন টব ব্যবহৃত হয়েছে।

# অতএব, মোট নতুন টব হবে:

# 18
# টি
# 18টি

# এখন, তার কাছে নতুন চারাগাছের টব থেকে কেমন অবশিষ্ট থাকবে তা বের করতে:

# টোটাল টব (80) থেকে 40টি টব দেওয়া হলে অবশিষ্ট টব হবে:

# 80
# −
# 40
# =
# 40
# টি টব
# 80−40=40টি টব

# মেরি 18টি নতুন টব দিয়েছিলেন।

# তাহলে,

# মেরির কাছে নতুন টবের অবশিষ্ট সংখ্যা হবে:

# 18
# টি টব
# 18টি টব

# তাহলে, মেরির কাছে 18টি নতুন চারাগাছের টব অবশিষ্ট থাকবে।

In [ ]:
# 독일어
llama3_1_inference_result = generate_response(system_message="",
                             user_message="Mary ist Gärtnerin aus Leidenschaft. Gestern hat sie 18 neue Topfpflanzen von ihrer Lieblingsgärtnerei bekommen. Sie hat bereits 2 Topfpflanzen auf jedem der 40 Fenstersimse ihres großen Landhauses stehen. Ihr ist großzügig zumute und sie hat sich entschlossen, dass sie 1 Topfpflanze von jedem Fenstersims morgen an Freunde und Familie verschenken möchte. Wie viele Topfpflanzen bleiben Mary übrig?",
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama3_1_inference_result)
# 정답 : 58

In [ ]:
# Llama 3.0

# Mary hat 18 neue Topfpflanzen erhalten. Sie hat bereits 2 Topfpflanzen auf jedem der 40 Fenstersimse. Das macht insgesamt 2 x 40 = 80 Topfpflanzen. Sie möchte 1 Topfpflanze von jedem Fenstersims verschenken, das sind weitere 40 Topfpflanzen. Insgesamt gibt sie also 80 + 40 = 120 Topfpflanzen weg. Sie hat 18 neue Topfpflanzen erhalten, also bleiben 18 - 120 = -102 Topfpflanzen übrig. Da es nicht möglich ist, negative Topfpflanzen zu haben, bleiben also 0 Topfpflanzen übrig.

In [ ]:
# GPT-4o

# Mary hat 18 neue Topfpflanzen bekommen. Momentan hat sie 2 Topfpflanzen auf jedem der 40 Fenstersimse, also insgesamt:

# 40
# ×
# 2
# =
# 80
#  Topfpflanzen
# 40×2=80 Topfpflanzen
# Nachdem sie 1 Topfpflanze von jedem Fenstersims verschenkt, gibt sie insgesamt:

# 40
# ×
# 1
# =
# 40
#  Topfpflanzen weg
# 40×1=40 Topfpflanzen weg
# Das bedeutet, sie behält:

# 80
# −
# 40
# =
# 40
#  Topfpflanzen
# 80−40=40 Topfpflanzen
# Dazu kommen die 18 neuen Pflanzen, die sie gestern bekommen hat:

# 40
# +
# 18
# =
# 58
#  Topfpflanzen
# 40+18=58 Topfpflanzen
# Mary bleiben also insgesamt 58 Topfpflanzen übrig.

In [ ]:
# Clova X

# Mary hat insgesamt 40 Fenstersimse * 2 Topfpflanzen = 80 Topfpflanzen.
# Sie möchte morgen 1 Topfpflanze von jedem Fenstersims verschenken, also insgesamt 40 Topfpflanzen.
# Daher bleiben ihr 80 - 40 = 40 Topfpflanzen übrig.

In [ ]:
# Gemini

# Lass uns das gemeinsam Schritt für Schritt berechnen:

# Wie viele Topfpflanzen hat Mary insgesamt, bevor sie verschenkt?

# Sie hatte 18 neue.
# Auf jedem der 40 Fenstersimse standen vorher schon 2. Das macht insgesamt 40 Fenstersimse * 2 Pflanzen/Fenstersims = 80 Pflanzen.
# Insgesamt: 18 neue + 80 alte = 98 Topfpflanzen.
# Wie viele Pflanzen verschenkt sie?

# Sie verschenkt 1 Pflanze von jedem der 40 Fenstersimse, also insgesamt 40 Pflanzen.
# Wie viele Pflanzen bleiben übrig?

# Insgesamt: 98 Pflanzen - 40 verschenkte Pflanzen = 58 Topfpflanzen.
# Antwort: Nach dem Verschenken bleiben Mary noch 58 Topfpflanzen übrig.

# Sie hat also immer noch eine große Anzahl an Pflanzen, um ihr Haus zu verschönern oder weitere Geschenke zu machen.

In [ ]:
# GPT-4o mini

# Um die Anzahl der übrig bleibenden Topfpflanzen zu berechnen, können wir die folgende Schritt-für-Schritt-Methode anwenden:

# Berechnung der ursprünglichen Anzahl der Topfpflanzen:
# Mary hat 2 Topfpflanzen auf jedem der 40 Fenstersimse. Daher beträgt die ursprüngliche Anzahl der Topfpflanzen:

# 2
#  Topfpflanzen/Fenstersims
# ×
# 40
#  Fenstersimse
# =
# 80
#  Topfpflanzen
# 2 Topfpflanzen/Fenstersims×40 Fenstersimse=80 Topfpflanzen
# Berücksichtigung der neuen Topfpflanzen:
# Mary hat 18 neue Topfpflanzen erhalten, daher ist die Gesamtanzahl der Topfpflanzen jetzt:

# 80
#  urspr
# u
# ¨
# ngliche Topfpflanzen
# +
# 18
#  neue Topfpflanzen
# =
# 98
#  Topfpflanzen
# 80 urspr
# u
# ¨
#  ngliche Topfpflanzen+18 neue Topfpflanzen=98 Topfpflanzen
# Berechnung der Topfpflanzen, die sie verschenken möchte:
# Mary plant, 1 Topfpflanze von jedem der 40 Fenstersimse zu verschenken. Das sind:

# 1
#  Topfpflanze/Fenstersims
# ×
# 40
#  Fenstersimse
# =
# 40
#  Topfpflanzen
# 1 Topfpflanze/Fenstersims×40 Fenstersimse=40 Topfpflanzen
# Berechnung der übrig bleibenden Topfpflanzen:
# Subtrahiere die verschenkte Anzahl von der Gesamtanzahl:

# 98
#  Topfpflanzen
# −
# 40
#  verschenkte Topfpflanzen
# =
# 58
#
# u
# ¨
# brig bleibende Topfpflanzen
# 98 Topfpflanzen−40 verschenkte Topfpflanzen=58
# u
# ¨
#  brig bleibende Topfpflanzen
# Also, Mary hat nach dem Verschenken 58 Topfpflanzen übrig.

In [ ]:
# 스페인어
llama3_1_inference_result = generate_response(system_message="",
                             user_message="Mary es una ferviente jardinera. Ayer, recibió 18 nuevas plantas de maceta de su vivero favorito. Ya tiene 2 plantas de maceta en cada una de las 40 repisas de las ventanas de su gran casa de campo. Como es generosa, ha decidido que mañana dará 1 planta de maceta de cada repisa a amigos y familiares. ¿Cuántas plantas de maceta le quedarán a Mary?",
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama3_1_inference_result)
# 정답 : 58

In [ ]:
# Llama 3.0

# Mary tiene 40 repisas y en cada una de ellas hay 2 plantas. Así que tiene 40 x 2 = 80 plantas. Luego, ella da 40 plantas a amigos y familiares. Así que le quedarán 80 - 40 = 40 plantas. Además, ayer recibió 18 plantas. Así que le quedarán 40 + 18 = 58 plantas.

In [ ]:
# GPT-4o

# Para resolver el problema, sigamos estos pasos:

# Calcular la cantidad inicial de plantas en las repisas:
# Mary tiene 2 plantas de maceta en cada una de las 40 repisas.

# 2
#  plantas/repisas
# ×
# 40
#  repisas
# =
# 80
#  plantas
# 2 plantas/repisas×40 repisas=80 plantas
# Sumar las nuevas plantas recibidas:
# Mary recibió 18 nuevas plantas de maceta.

# 80
#  plantas
# +
# 18
#  plantas
# =
# 98
#  plantas en total
# 80 plantas+18 plantas=98 plantas en total
# Restar las plantas que dará a amigos y familiares:
# Mary dará 1 planta de maceta de cada repisa, y hay 40 repisas.

# 98
#  plantas
# −
# 40
#  plantas
# =
# 58
#  plantas
# 98 plantas−40 plantas=58 plantas
# Mary tendrá 58 plantas de maceta después de haber dado una de cada repisa a sus amigos y familiares.

In [ ]:
# Clova X

# Mary tiene 2 plantas de maceta en cada una de las 40 repisas, lo que significa que tiene un total de 2 x 40 = 80 plantas de maceta.

# Ayer recibió 18 nuevas plantas de maceta, por lo que ahora tiene 80 + 18 = 98 plantas de maceta en total.

# Mañana dará 1 planta de maceta de cada repisa, lo que significa que dará 40 plantas de maceta.

# Después de dar las plantas, le quedarán 98 - 40 = 58 plantas de maceta.

In [ ]:
# Gemini

# ¡Claro que sí! Vamos a resolver este problema paso a paso para saber cuántas plantas le quedarán a Mary.

# 1. Plantas que tiene ahora:

# Nuevas plantas: 18
# Plantas en repisas: 2 plantas/repisa * 40 repisas = 80 plantas
# Total actual: 18 + 80 = 98 plantas
# 2. Plantas que dará:

# Plantas por repisa a regalar: 1 planta/repisa
# Total de repisas: 40
# Plantas a regalar en total: 1 planta/repisa * 40 repisas = 40 plantas
# 3. Plantas que le quedarán:

# Plantas totales - plantas a regalar: 98 plantas - 40 plantas = 58 plantas
# Respuesta: A Mary le quedarán 58 plantas de maceta.

# ¡Así que después de su generoso regalo, Mary todavía tendrá muchas plantas para disfrutar en su jardín!

In [ ]:
# GPT-4o mini

# Para resolver cuántas plantas de maceta le quedarán a Mary después de regalar algunas, sigamos estos pasos:

# Calcular el número total de plantas que Mary tiene inicialmente:

# Mary tiene 2 plantas de maceta en cada una de las 40 repisas.
# Entonces, el número total de plantas es:
# 2
#  plantas/repisas
# ×
# 40
#  repisas
# =
# 80
#  plantas
# 2 plantas/repisas×40 repisas=80 plantas.
# Sumar las 18 nuevas plantas que recibió:

# El total de plantas ahora es:
# 80
#  plantas
# +
# 18
#  plantas
# =
# 98
#  plantas
# 80 plantas+18 plantas=98 plantas.
# Calcular cuántas plantas regalará Mary:

# Mary regalará 1 planta de maceta de cada repisa.
# Como tiene 40 repisas, regalará un total de:
# 40
#  plantas
# 40 plantas.
# Calcular cuántas plantas le quedarán a Mary después de regalar:

# El número de plantas restantes es:
# 98
#  plantas
# −
# 40
#  plantas
# =
# 58
#  plantas
# 98 plantas−40 plantas=58 plantas.
# Por lo tanto, a Mary le quedarán 58 plantas de maceta después de regalar algunas.

In [ ]:
# 프랑스어
llama3_1_inference_result = generate_response(system_message="",
                             user_message="Marie adore jardiner. Hier, elle a reçu 18 nouvelles plantes en pot provenant de sa pépinière préférée. Elle en avait déjà 2 sur chacun des 40 rebords de fenêtre de sa grande maison de campagne. Elle a donc décidé que demain, elle prendra sur chaque rebord de fenêtre 1 plante en pot qu'elle offrira généreusement à des amis et à des proches. Combien de plantes en pot restera-t-il à Marie ?",
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama3_1_inference_result)
# 정답 : 58

In [ ]:
# Llama 3.0

# Marie a 40 rebords de fenêtre et 2 plantes en pot sur chaque rebord. Elle en aura donc 40 x 2 = 80 plantes en pot. Elle reçoit 18 nouvelles plantes en pot. Elle en offre 40 (1 par rebord) et elle aura donc 80 + 18 - 40 = 58 plantes en pot restantes.

In [ ]:
# GPT-4o

# Marie a initialement 2 plantes en pot sur chacun des 40 rebords de fenêtre. Le nombre total de plantes en pot qu'elle possède déjà est donc :

# 2
#  plantes
# ×
# 40
#  rebords de fen
# e
# ˆ
# tre
# =
# 80
#  plantes
# 2 plantes×40 rebords de fen
# e
# ˆ
#  tre=80 plantes
# Elle a ensuite reçu 18 nouvelles plantes en pot. Le nombre total de plantes en pot qu'elle possède maintenant est :

# 80
#  plantes
# +
# 18
#  nouvelles plantes
# =
# 98
#  plantes
# 80 plantes+18 nouvelles plantes=98 plantes
# Marie décide de prendre 1 plante en pot sur chacun des 40 rebords de fenêtre pour les offrir. Elle va donc retirer 40 plantes en pot. Le nombre de plantes en pot qu'il lui restera est :

# 98
#  plantes
# −
# 40
#  plantes offertes
# =
# 58
#  plantes
# 98 plantes−40 plantes offertes=58 plantes
# Il restera donc 58 plantes en pot à Marie.

In [ ]:
# Clova X

# Marie a reçu 18 nouvelles plantes en pot et en avait déjà 2 sur chaque rebord de fenêtre, soit un total de 40 * 2 = 80 plantes en pot.
# Elle décide de prendre 1 plante en pot sur chaque rebord de fenêtre pour les offrir à des amis et à des proches, donc elle en donnera 40.
# Il lui restera donc 80 - 40 = 40 plantes en pot.

In [ ]:
# Gemini

# Absolument ! Analysons cela ensemble.

# Plantes au départ:

# Nouvelles plantes: 18
# Plantes déjà sur les rebords: 2 plantes/rebord * 40 rebords = 80 plantes
# Total initial: 18 + 80 = 98 plantes

# Plantes offertes:

# Marie offre 1 plante/rebord * 40 rebords = 40 plantes
# Plantes restantes:

# 98 plantes initiales - 40 plantes offertes = 58 plantes
# Réponse:

# Il restera donc 58 plantes en pot à Marie après avoir fait ses offrandes.

# Elle aura toujours un magnifique jardin !

In [ ]:
# GPT-4o mini

# Pour déterminer combien de plantes en pot resteraient à Marie après qu'elle en aura donné une sur chaque rebord de fenêtre, procédons comme suit :

# Calcul du nombre total de plantes en pot avant le don :

# Marie avait 2 plantes sur chacun des 40 rebords de fenêtre. Le nombre total de plantes en pot est donc :

# 2
#  plantes/rebord
# ×
# 40
#  rebords
# =
# 80
#  plantes
# 2 plantes/rebord×40 rebords=80 plantes
# Elle a reçu 18 nouvelles plantes, donc le nombre total de plantes devient :

# 80
#  plantes
# +
# 18
#  plantes
# =
# 98
#  plantes
# 80 plantes+18 plantes=98 plantes
# Nombre de plantes données :

# Marie décide de donner 1 plante de chaque rebord de fenêtre. Étant donné qu'il y a 40 rebords de fenêtre, elle donnera :

# 1
#  plante/rebord
# ×
# 40
#  rebords
# =
# 40
#  plantes
# 1 plante/rebord×40 rebords=40 plantes
# Calcul des plantes restantes :

# Pour trouver combien de plantes resteront après avoir donné 40 plantes, soustrayons le nombre de plantes données du nombre total initial :

# 98
#  plantes
# −
# 40
#  plantes
# =
# 58
#  plantes
# 98 plantes−40 plantes=58 plantes
# Donc, après avoir offert 1 plante sur chaque rebord de fenêtre, il restera à Marie 58 plantes en pot.

In [ ]:
# 러시아어
llama3_1_inference_result = generate_response(system_message="",
                             user_message="Мэри — заядлый садовод. Вчера она получила 18 новых горшечных растений из своего любимого питомника. У нее уже есть по 2 горшечных растения на каждом из 40 подоконников в ее большом загородном доме. От щедрости она решила, что завтра отдаст по 1 горшечному растению с каждого подоконника друзьям и родственникам. Сколько горшечных растений останется у Мэри?",
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama3_1_inference_result)
# 정답 : 58

In [ ]:
# Llama 3.0

# Если она отдаст по 1 горшечному растению с каждого подоконника, то это будет 40 горшечных растений. Она отдаст 40 горшечных растений, оставив у себя 98 - 40 = 58 горшечных растений.

In [ ]:
# GPT-4o

# У Мэри изначально было по 2 горшечных растения на каждом из 40 подоконников, то есть всего:

# 2
# ×
# 40
# =
# 80
#  горшечных растений
# 2×40=80 горшечных растений
# Вчера она получила 18 новых растений, значит, у нее стало:

# 80
# +
# 18
# =
# 98
#  горшечных растений
# 80+18=98 горшечных растений
# Завтра она собирается отдать по 1 растению с каждого подоконника, а у нее 40 подоконников. Значит, она отдаст:

# 40
#  горшечных растений
# 40 горшечных растений
# Таким образом, у нее останется:

# 98
# −
# 40
# =
# 58
#  горшечных растений
# 98−40=58 горшечных растений
# Ответ: у Мэри останется 58 горшечных растений.

In [ ]:
# Clova X

# У Мэри есть 40 подоконников, на каждом из которых растет по 2 горшечных растения, то есть всего 40 * 2 = 80 горшечных растений.

# Она решила отдать по 1 горшечному растению с каждого подоконника, поэтому отдаст 40 * 1 = 40 горшечных растений.

# Итого, у Мэри останется 80 - 40 = 40 горшечных растений.

In [ ]:
# Gemini

# Давайте разберемся с этой задачей вместе.

# Что мы знаем:

# У Мэри было 40 подоконников.
# На каждом подоконнике было по 2 растения.
# Мэри получила еще 18 растений.
# Она отдала по 1 растению с каждого подоконника.
# Сначала посчитаем, сколько растений было у Мэри до того, как она получила новые:

# 40 подоконников * 2 растения/подоконник = 80 растений

# Теперь добавим новые растения:

# 80 растений + 18 новых растений = 98 растений

# Затем вычтем растения, которые она отдала:

# 98 растений - 40 подоконников * 1 растение/подоконник = 98 растений - 40 растений = 58 растений

# Ответ: После всех этих манипуляций у Мэри осталось 58 горшечных растений.

# Другой способ решения:

# Можно сразу посчитать, сколько растений останется на каждом подоконнике после того, как Мэри отдаст одно:

# 2 растения/подоконник - 1 растение/подоконник = 1 растение/подоконник

# Затем умножить это число на количество подоконников и прибавить новые растения:

# 40 подоконников * 1 растение/подоконник + 18 новых растений = 58 растений

# Оба способа приводят к одному и тому же ответу: у Мэри осталось 58 горшечных растений.

In [ ]:
# GPT-4o mini

# Чтобы найти, сколько горшечных растений останется у Мэри, давайте проведем расчеты шаг за шагом:

# Определим количество горшечных растений, которые у Мэри уже есть.

# У нее есть 40 подоконников.
# На каждом подоконнике по 2 горшечных растения.
# Таким образом, общее количество горшечных растений на подоконниках:
# 40
# ×
# 2
# =
# 80
# 40×2=80.
# Добавим новые горшечные растения, которые Мэри получила.

# Мэри получила 18 новых горшечных растений.
# Следовательно, общее количество горшечных растений у Мэри сейчас:
# 80
# +
# 18
# =
# 98
# 80+18=98.
# Определим, сколько горшечных растений Мэри отдаст.

# Мэри отдаст по 1 горшечному растению с каждого подоконника.
# Поскольку у нее 40 подоконников, она отдаст 40 горшечных растений.
# Вычислим, сколько горшечных растений останется у Мэри.

# Начальное количество растений: 98.
# Количество растений, которое она отдаст: 40.
# Количество оставшихся растений:
# 98
# −
# 40
# =
# 58
# 98−40=58.
# Итак, после того как Мэри отдаст по 1 горшечному растению с каждого подоконника, у нее останется 58 горшечных растений.

In [ ]:
# 스와힐리어
llama3_1_inference_result = generate_response(system_message="",
                             user_message="Mary mkulima wa bustani mwenye shauku. Jana, alipokea mimea 18 iliyowekwa kwenye nyungu kutoka kwa bustani ya miche anayopendelea zaidi. Tayari ana mimea 2 kwenye nyungu kwenye kila vishubaka 40 vya dirisha lake katika boma lake kubwa la mashambani. Kwa kuwa na ukarimu, ameamua kuwa kesho atapeana mmea 1 ulio kwenye chungu kutoka kwa kila kishubaka kwa marafiki na familia. Mary atasalia na mimea mingapi iliyo kwenye nyungu?",
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama3_1_inference_result)
# 정답 : 58

In [ ]:
# Llama 3.0

# Let's break down the problem step by step:

# 1. Mary received 18 new plants from her favorite garden.
# 2. She already had 2 plants in each of the 40 compartments of her large farm shed, so she had a total of 2 x 40 = 80 plants.
# 3. She decided to give away 1 plant from each compartment to her friends and family tomorrow, so she will give away 1 x 40 = 40 plants.

# To find out how many plants Mary will have left, we need to subtract the number of plants she gave away from the total number of plants she had:

# 80 (initial plants) + 18 (new plants) - 40 (plants given away) = 58

# So, Mary will have 58 plants left.

In [ ]:
# GPT-4o

# Mary ana mimea 2 kwenye nyungu kwenye kila kishubaka 40 cha dirisha lake. Hii inamaanisha kwa jumla ana mimea:

# 2
# ×
# 40
# =
# 80
#  mimea
# 2×40=80 mimea
# Alipokea mimea mingine 18 jana, kwa hivyo kwa sasa ana jumla ya mimea:

# 80
# +
# 18
# =
# 98
#  mimea
# 80+18=98 mimea
# Kesho, Mary atapeana mmea 1 kutoka kwa kila kishubaka kwa marafiki na familia. Ana vishubaka 40, kwa hiyo atatoa mimea:

# 1
# ×
# 40
# =
# 40
#  mimea
# 1×40=40 mimea
# Baada ya kutoa mimea hii, Mary atasalia na:

# 98
# −
# 40
# =
# 58
#  mimea
# 98−40=58 mimea
# Kwa hivyo, Mary atasalia na mimea 58 iliyo kwenye nyungu.

In [ ]:
# Clova X

# Mary ana mimea 2 kwenye nyungu kwenye kila vishubaka 40 vya dirisha lake katika boma lake kubwa la mashambani. Kwa hiyo, yote jumla ya mimea iliyo kwenye nyungu ni 2 x 40 = 80 mimea.

# Jana, alipokea mimea 18 iliyowekwa kwenye nyungu kutoka kwa bustani ya miche anayopendelea zaidi. Kwa hiyo, mimea iliyo kwenye nyungu yake sasa ni 80 + 18 = 98 mimea.

# Kesho, atapeana mmea 1 ulio kwenye chungu kutoka kwa kila kishubaka kwa marafiki na familia. Kwa hiyo, mimea iliyo kwenye nyungu yake baada ya kupoteza mmea kwa marafiki ni 98 - 1 = 97 mimea.

In [ ]:
# Gemini

# Haya, tufanye hesabu pamoja!

# Hatua za kutatua:

# Mimea yote Mary aliyonayo mwanzoni:

# Mimea mpya kutoka kwenye bustani ya miche: 18
# Mimea alizonazo tayari kwenye vishubaka:
# Vishubaka vyote: 40
# Mimea kwenye kila kishubaka: 2
# Jumla ya mimea aliyonayo tayari: 40 vishubaka * 2 mimea/kishubaka = 80 mimea
# Jumla ya mimea yote: 18 + 80 = 98 mimea
# Mimea atakayotoa:

# Atapeana mmea 1 kutoka kila kishubaka 40: 40 mimea
# Mimea itakayobaki:

# Mimea yote - Mimea atakayotoa = 98 mimea - 40 mimea = 58 mimea
# Jibu:
# Baada ya kutoa mimea, Mary atasalia na mimea 58 iliyo kwenye nyungu.

# Kwa kifupi:
# Mary atakuwa na mimea 58 baada ya kutoa zawadi.

# Je, ungependa kuuliza swali lingine?

# Kwa mfano, unaweza kuuliza ni vishubaka vingapi vitabaki na mmea mmoja baada ya kutoa zawadi.

In [ ]:
# GPT-4o mini

# Mary ana mimea 2 kwenye nyungu kwenye kila vishubaka 40 vya dirisha lake. Kwa hivyo, jumla ya mimea kwenye nyungu ni:

# 2
#  mimea/vishubaka
# ×
# 40
#  vishubaka
# =
# 80
#  mimea
# 2 mimea/vishubaka×40 vishubaka=80 mimea

# Mary alipokea mimea 18 zaidi jana, hivyo jumla ya mimea kwenye nyungu sasa ni:

# 80
#  mimea
# +
# 18
#  mimea
# =
# 98
#  mimea
# 80 mimea+18 mimea=98 mimea

# Kesho, Mary atapeana mmea 1 ulio kwenye chungu kutoka kwa kila kishubaka kwa marafiki na familia. Kwa hivyo, atatoa:

# 1
#  mmea/vishubaka
# ×
# 40
#  vishubaka
# =
# 40
#  mimea
# 1 mmea/vishubaka×40 vishubaka=40 mimea

# Hivyo, mimea itakayobaki kwenye nyungu baada ya kutoa ni:

# 98
#  mimea
# −
# 40
#  mimea
# =
# 58
#  mimea
# 98 mimea−40 mimea=58 mimea

# Mary atasalia na mimea 58 iliyo kwenye nyungu baada ya kutoa mimea kwa marafiki na familia.

In [ ]:
# 텔루구어
llama3_1_inference_result = generate_response(system_message="",
                             user_message="మేరీ ఎంతో ఆసక్తి కలిగిన తోటిమాలి. ఆమె తనకు ఇష్టమైన మొక్కల నర్సరీ నుంచి 18 కొత్త కుండీ మొక్కలను అందుకుంది. ఆమె పెద్ద దేశీయ గృహంలోని 40 కిటికీ అంచుల్లో ఒక్కొక్కదానికి 2 కుండీ మొక్కలను ఇప్పటికే ఉంచింది. ఉదారతను ప్రదర్శిస్తూ, ఆమె ప్రతి లెడ్జ్ నుంచి 1 కుండీ మొక్కను స్నేహితులు మరియు కుటుంబాలకు రేపు ఇవ్వాలని నిర్ణయించుకుంది. మెరీ వద్ద ఎన్ని కుండీ మొక్కలు ఉంటాయి?",
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama3_1_inference_result)
# 정답 : 58

In [ ]:
# Llama 3.0

# Let's break down the problem step by step:

# 1. Mary got 18 new plants from the nursery.
# 2. She already had 2 plants per window, and there are 40 windows, so she had a total of 2 x 40 = 80 plants.
# 3. She decided to give 1 plant per ledge to her friends and family, and there are 40 ledges, so she will give away 40 plants.
# 4. To find the number of plants Mary has left, we subtract the number of plants she gave away (40) from the total number of plants she had (80 + 18 = 98):

# 98 - 40 = 58

# So, Mary has 58 plants left.

# Note that the problem doesn't specify what happened to the remaining 18 plants, but since Mary gave away 40 plants, it's likely that the remaining 18 plants are the ones she didn't give away.

In [ ]:
# GPT-4o

# మేరీ మొత్తం 18 కొత్త కుండీ మొక్కలను నర్సరీ నుంచి అందుకుంది.

# మేరీ 40 కిటికీ అంచుల్లో ఒక్కొక్కదానికి 2 కుండీ మొక్కలను ఉంచింది, అంటే మొత్తం 40 × 2 = 80 కుండీ మొక్కలు.

# ఇప్పుడు ఆమె ప్రతీ కిటికీ అంచు నుంచి 1 కుండీ మొక్కను స్నేహితులు మరియు కుటుంబాలకు ఇవ్వాలని నిర్ణయించింది, అంటే 40 కిటికీ అంచుల్లో నుండి 40 కుండీ మొక్కలను ఇవ్వనుంది.

# అయితే, ఆమె వద్ద ఉన్న మొత్తం కుండీ మొక్కలు = 80 (ఇప్పటికే ఉంచినవి) + 18 (కొత్తగా పొందినవి) = 98.

# ఆమె 40 కుండీ మొక్కలను ఇవ్వడం ద్వారా, ఆమె వద్ద మిగిలి ఉన్న కుండీ మొక్కల సంఖ్య = 98 - 40 = 58.

# కాబట్టి, చివరికి మెరీ వద్ద 58 కుండీ మొక్కలు మిగులుతాయి.

In [ ]:
# Clova X

# మనం ఈ సమస్యను కలిసి పరిష్కరిద్దాం.
# మొదటి భాగం: మేరీ ఇప్పుడు ఎన్ని మొక్కలు కలిగి ఉంది?
# కొత్త మొక్కలు: 18
# ఇప్పటికే ఉన్న మొక్కలు: 40 కిటికీలు * 2 మొక్కలు/కిటికీ = 80 మొక్కలు
# మొత్తం: 18 + 80 = 98 మొక్కలు
# రెండవ భాగం: మేరీ ఎన్ని మొక్కలు ఇస్తుంది?
# కిటికీలు: 40
# ప్రతి కిటికీ నుండి ఇచ్చే మొక్కలు: 1
# మొత్తం ఇచ్చే మొక్కలు: 40 * 1 = 40 మొక్కలు
# మూడవ భాగం: మేరీ వద్ద మిగిలిన మొక్కలు
# మొదటి నుండి మొక్కలు: 98
# ఇచ్చే మొక్కలు: 40
# మిగిలిన మొక్కలు: 98 - 40 = 58 మొక్కలు
# సమాధానం: మేరీ వద్ద 58 కుండీ మొక్కలు మిగిలుంటాయి.

# వివరణ: మేరీ తన కొత్త మొక్కలతో పాటు ఇప్పటికే ఉన్న మొక్కలను కలిపి, తర్వాత తన స్నేహితులకు మరియు కుటుంబాలకు ఇచ్చే మొక్కలను తీసివేయడం ద్వారా మనం ఈ సమస్యను పరిష్కరించాము.

# ఇది సరైన సమాధానమా? అదనపు ప్రశ్నలు ఉన్నాయా?

In [ ]:
# Gemini

# చాలా బాగుంది, ఈ కథను సరిగ్గా అర్థం చేసుకున్నాను.
# మనం ఈ సమస్యను కలిసి పరిష్కరిద్దాం.

# మొదట మనకు తెలిసిన విషయాలు ఇవే:

# మేరీకి 18 కొత్త మొక్కలు వచ్చాయి.
# ఆమె ఇంట్లో 40 కిటికీలు ఉన్నాయి.
# ప్రతి కిటికీలో ఇప్పటికే 2 మొక్కలు ఉన్నాయి.
# ఆమె ప్రతి కిటికీ నుండి ఒక మొక్కను ఇవ్వాలనుకుంటుంది.
# ఇప్పుడు మనం ఈ సమస్యను దశల వారీగా పరిష్కరిద్దాం:

# మేరీ ఇప్పటికే ఎన్ని మొక్కలు కలిగి ఉంది?

# ప్రతి కిటికీలో 2 మొక్కలు ఉన్నాయి.
# 40 కిటికీలు ఉన్నాయి.
# కాబట్టి, ఆమె ఇప్పటికే కలిగి ఉన్న మొక్కలు = 40 కిటికీలు * 2 మొక్కలు/కిటికీ = 80 మొక్కలు.
# మేరీకి ఇప్పుడు ఎన్ని మొక్కలు ఉన్నాయి?

# ఆమె ఇప్పటికే ఉన్న మొక్కలు + కొత్త మొక్కలు = 80 మొక్కలు + 18 మొక్కలు = 98 మొక్కలు.
# ఆమె ఎన్ని మొక్కలు ఇస్తుంది?

# ఆమె 40 కిటికీలు ఉన్నాయి మరియు ప్రతి కిటికీ నుండి ఒక మొక్కను ఇస్తుంది.
# కాబట్టి, ఆమె 40 మొక్కలు ఇస్తుంది.
# మేరీ వద్ద చివరకు ఎన్ని మొక్కలు మిగిలుంటాయి?

# మొత్తం మొక్కలు - ఇచ్చిన మొక్కలు = 98 మొక్కలు - 40 మొక్కలు = 58 మొక్కలు.
# కాబట్టి, మేరీ వద్ద చివరకు 58 కుండీ మొక్కలు మిగిలుంటాయి.

# సరే, ఇప్పుడు నువ్వు ప్రయత్నించు. ఈ సమస్యను నీ స్వంత మాటల్లో వివరించగలవా?

In [ ]:
# GPT-4o mini

# మేరీ వద్ద ప్రస్తుతం ఉన్న కుండీ మొక్కల సంఖ్యను తెలుసుకోవాలంటే, ఆమె ఈ ప్రక్రియను అనుసరించాలి:

# ప్రారంభ కుండీ మొక్కల సంఖ్య: 18
# ప్రస్తుతం ఉంచిన కుండీ మొక్కలు: 40 కిటికీ అంచుల్లో ఒక్కొక్కదానికి 2 మొక్కలు అంటే, మొత్తం 40 × 2 = 80 కుండీ మొక్కలు ఉన్నాయి.
# ప్రతి లెడ్జ్ నుంచి ఇవ్వాల్సిన కుండీ మొక్కలు: 40 లెడ్జ్‌లకు ఒక్కొక్కదానికీ 1 మొక్క ఇవ్వాలని నిర్ణయించింది, అంటే మొత్తం 40 × 1 = 40 మొక్కలు ఇవ్వాలి.
# ఇప్పుడు, ముందుగా ఉన్న 80 కుండీ మొక్కలలో 40 మొక్కలు ఇచ్చిన తరువాత:

# 80
# −
# 40
# =
# 40
# 80−40=40

# అంతే కాదు, మెరీ తోటిమాలి వద్ద ఇంకా ఉన్న 18 కొత్త కుండీ మొక్కలు కూడా ఉన్నాయి. అవి మొత్తం కుండీ మొక్కలలో కలిపితే:

# 40
# +
# 18
# =
# 58
# 40+18=58

# కాబట్టి, మెరీ వద్ద ప్రస్తుతం 58 కుండీ మొక్కలు ఉంటాయి.

# 추론 정확도에 대한 정량적 분석

|모델|한국어|영어|일본어|중국어|태국어|벵골어|독일어|스페인어|프랑스어|러시아어|스와힐리어|텔루구어|
|------|---|---|---|---|---|---|---|---|---|---|---|---|
|Llama3.1-8B-Instruct|o|o|o|o|x|x|x|o|o|o|o|x|
|Llama3.0-8B-Instruct|o(영어답변)|o|o(영어답변)|x|x|x|x|o|o|o|o(영어답변)|o(영어답변)|
|GPT-4o|o|o|o|o|o|o|o|o|o|o|o|o|
|Clova X|o|x|x|x|x|x|x|o|x|x|x|o|
|Gemini|o|o|o|o|x|o|o|o|o|o|o|o|
|GPT-4o mini|o|o|o|o|o|x|o|o|o|o|o|o|

In [ ]:
# 맞춘 갯수 분석
# 1위 : GPT-4o - 12/12
# 2위 : GPT-4o mini, Gemini - 11/12
# 4위 : Llama 3.1-8B-Instruct, Llama 3.0-8B-Instruct - 8/12
# 6위 : Clova X - 3/12